In [ ]:
%cd ../
%ls

In [ ]:
from wag_toolkit.locations import Locations
import pandas as pd
import json
from collections import Counter
from dotenv import load_dotenv
import awswrangler as wr
import numpy as np

# Load environment variables from .env file
load_dotenv()

In [ ]:
# grants_filename = 'funding_impact_measures/whole_portfolio/canonical_reference_data/20240113_uk_funder_data/dimcli_grants_2019_2023.parquet'
pubs_filename = 'funding_impact_measures/whole_portfolio/canonical_reference_data/20240113_uk_funder_data/dimcli_pubs_2018_2022.parquet'
 
# grants_df = wr.s3.read_parquet(f's3://datalabs-data/{grants_filename}') 
pubs_df = wr.s3.read_parquet(f's3://datalabs-data/{pubs_filename}')

# funders = ['Wellcome Trust', 'Medical Research Council']

In [ ]:
pubs_df = pubs_df[pubs_df['funder']=='Wellcome Trust']
pub_ids = list(pubs_df['id'].unique())

In [ ]:
from wag_toolkit.institutions import Institutions

In [ ]:
dummy_query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication) RETURN * LIMIT 1"""

inst = Institutions(dummy_query)
query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication)
            WHERE p.dimensions_publication_id IN {}
            RETURN p.dimensions_publication_id AS dimensions_publication_id, p.year AS year,
                a.institutions AS grid_id"""
inst.lookup_query(query=query, lookup=pub_ids)

In [ ]:
data =pd.DataFrame(inst.data)
data = data.explode("grid_id").dropna()
data.head()

In [ ]:
inst._clean_grid_ids()
inst.extract_edges()
inst.extract_locations('institution')
inst.convert_edges()
inst.calculate_adjacency_matrices()

In [ ]:
inst.extract_networks()
inst.calculate_centrality()

In [ ]:
G = inst.G[2018]

In [ ]:
import community as community_louvain  # pip install python-louvain
import networkx as nx

partition = community_louvain.best_partition(G)

density = nx.density(G)

# For numeric node attribute 'size':
assortativity = nx.attribute_assortativity_coefficient(G, 'size')


# avg_path_length = nx.average_shortest_path_length(G)
# diameter = nx.diameter(G)


In [ ]:
inst.find_louvain_communities()

In [ ]:
import community as community_louvain

modularity = community_louvain.modularity(partition, G)
print(f"Modularity score: {modularity:.4f}")


In [ ]:
inst.plot_louvain_communities()

In [ ]:
import plotly.graph_objects as go
import networkx as nx
import plotly.express as px

# Compute layout positions
pos = nx.spring_layout(G, seed=42)  # deterministic layout

# Create edge traces
edge_x = []
edge_y = []
for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines'
)

# Prepare discrete colors for communities
num_communities = max(partition.values()) + 1
colors = px.colors.qualitative.Safe  # qualitative palette with distinct colors
palette = colors * (num_communities // len(colors) + 1)  # repeat if needed

# Create node traces per community for legend
node_traces = []
for community_id in range(num_communities):
    node_x = []
    node_y = []
    node_text = []

    for node in G.nodes():
        if partition[node] == community_id:
            x, y = pos[node]
            node_x.append(x)
            node_y.append(y)
            node_text.append(node)

    trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        text=node_text,
        name=f'Community {community_id}',
        marker=dict(
            color=palette[community_id],
            size=10,
            line_width=0.5
        )
    )
    node_traces.append(trace)

# Plot
fig = go.Figure(data=[edge_trace] + node_traces,
                layout=go.Layout(
                    title='Louvain Communities (Interactive)',
                    showlegend=True,
                    hovermode='closest',
                    margin=dict(b=20,l=5,r=5,t=40),
                    annotations=[dict(
                        text="Use zoom and hover for details",
                        showarrow=False,
                        xref="paper", yref="paper",
                        x=0.005, y=-0.002
                    )],
                    xaxis=dict(showgrid=False, zeroline=False),
                    yaxis=dict(showgrid=False, zeroline=False)
                )
)

fig.show()


In [ ]:
import plotly.graph_objects as go
import networkx as nx
import plotly.express as px

G_dict = inst.G  # Dictionary of graphs for each year
partition_dict = inst.louvain_communities  # Dictionary of DataFrames for each year

years = sorted(G_dict.keys())

fig = go.Figure()

# Add traces for each year, only first year's traces visible initially
for i, year in enumerate(years):
    G = G_dict[year]
    partition_df = partition_dict[year]
    partition = dict(zip(partition_df['institution'], partition_df['community']))

    pos = nx.spring_layout(G, seed=42)

    # Edge traces
    edge_x = []
    edge_y = []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines',
        visible=(i == 0)  # Only first year visible initially
    )
    fig.add_trace(edge_trace)

    # Node traces per community (for legend)
    num_communities = partition_df['community'].max() + 1
    colors = px.colors.qualitative.Safe
    palette = colors * (num_communities // len(colors) + 1)

    for community_id in range(num_communities):
        node_x = []
        node_y = []
        node_text = []
        for node in G.nodes():
            if partition.get(node) == community_id:
                x, y = pos[node]
                node_x.append(x)
                node_y.append(y)
                node_text.append(node)

        node_trace = go.Scatter(
            x=node_x, y=node_y,
            mode='markers',
            hoverinfo='text',
            text=node_text,
            name=f'Community {community_id}',
            marker=dict(
                color=palette[community_id],
                size=10,
                line_width=0.5
            ),
            visible=(i == 0)  # Only visible for first year initially
        )
        fig.add_trace(node_trace)

# Create buttons for dropdown menu to toggle year visibility
buttons = []
trace_index = 0
year_trace_indices = {}

for year in years:
    partition_df = partition_dict[year]
    n_communities = partition_df['community'].max() + 1

    # Indices of traces belonging to this year:
    indices = [trace_index]  # edge trace
    indices += list(range(trace_index + 1, trace_index + 1 + n_communities))
    year_trace_indices[year] = indices
    trace_index += 1 + n_communities

for year in years:
    vis = [False] * len(fig.data)
    for idx in year_trace_indices[year]:
        vis[idx] = True
    buttons.append(dict(
        label=str(year),
        method="update",
        args=[{"visible": vis},
              {"title": f"Louvain Communities (Interactive) - Year {year}"}]
    ))

# Add dropdown menu to layout
fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0,
        y=1.1,
        xanchor='left',
        yanchor='top'
    )],
    showlegend=True,
    hovermode='closest',
    margin=dict(b=20, l=5, r=5, t=40),
    annotations=[dict(
        text="Use zoom and hover for details",
        showarrow=False,
        xref="paper", yref="paper",
        x=0.005, y=-0.002
    )],
    xaxis=dict(showgrid=False, zeroline=False),
    yaxis=dict(showgrid=False, zeroline=False)
)

fig.show()
